## 0.准备

In [37]:
# 引入常用科学计算与绘图库
import sklearn
import matplotlib as mpl
import matplotlib.pyplot as plt
# 显示中文
mpl.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # 设置中文字体，防止中文乱码
mpl.rcParams['axes.unicode_minus'] = False            # 解决负号显示为方块的问题
# Jupyter 环境内嵌绘图
%matplotlib inline                                     
# 引入数据处理与深度学习相关库
import numpy as np
import pandas as pd
import os
import sys
import time
import tqdm.auto as tqdm   # 进度条库
import torch
import torch.nn as nn
import torch.nn.functional as F

# 打印 Python 及各主要库版本信息，方便调试与复现
print(sys.version_info)
for module in mpl,np,pd,sklearn,torch:
    print(module.__name__, module.__version__)

# 检测并设置计算设备（优先使用 GPU）
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# 固定随机种子，保证实验可复现
seed=42
torch.manual_seed(seed)      # 为 CPU 设置种子
torch.cuda.manual_seed_all(seed)  # 为所有 GPU 设置种子
np.random.seed(seed)         # 为 numpy 设置种子


sys.version_info(major=3, minor=11, micro=8, releaselevel='final', serial=0)
matplotlib 3.8.4
numpy 1.26.4
pandas 2.2.2
sklearn 1.4.2
torch 2.3.1
cpu


In [ ]:
# #   下载数据集
# import keras
# import pathlib
# import os
# # 设置下载到当前目录
# cache_dir = os.getcwd()
# text_file = keras.utils.get_file(
#     fname="spa-eng.zip",
#     origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
#     extract=True,
#     cache_dir=cache_dir,
# )
# text_file = pathlib.Path(text_file) / "spa-eng" / "spa.txt"

# 1.数据加载

In [ ]:
import unicodedata
import re
from sklearn.model_selection import train_test_split


# 应为西班牙语有一些特殊字符,所以我们需要unicode转ascii,
# 这样值变小了,因为unicode太大
def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


# test
en_sentence = "May I borrow this   book?"
sq_sentence = "¿Qué hora es?"
print(unicode_to_ascii(en_sentence))
print(unicode_to_ascii(sq_sentence))

May I borrow this   book?
¿Que hora es?


In [ ]:
def preprocess_sentence(w):
    # 转换为小写并删除前导和尾随空格
    w = unicode_to_ascii(w.lower().strip())

    # 在单词与跟在其后的标点之间插入一个空格
    # 例如： "he is a boy." => "he is a boy ."
    # 参考：https://stackoverflow.com/questions/3645931/python-padding-punctuation-with-white-spaces-keeping-punctuation
    w = re.sub(r"([?.!,¿])", r" \1 ", w)

    # 除了 (a-z, A-Z, ".", "?", "!", ",")，将所有字符替换为空格
    """^表示非,除了(a-z, A-Z, ".", "?", "!", ","),将所有字符替换为空格"""
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)

    # 因为可能有多余空格,替换为一个空格
    w = re.sub(r"[ ]+", " ", w)

    # 去掉句子前后的空格
    w = w.strip()

    # 给句子添加一个开始和一个结束标记
    # 以便模型知道何时开始和结束预测
    w = "<start> " + w + " <end>"
    return w


print(preprocess_sentence(en_sentence))
print(preprocess_sentence(sq_sentence))
print(preprocess_sentence(sq_sentence).encode("utf-8"))  # 编码为utf-8

<start> may i borrow this book ? <end>
<start> ¿ que hora es ? <end>
b'<start> \xc2\xbf que hora es ? <end>'


In [ ]:
# 2.数据划分
# 我们将数据集随机划分成训练集和测试集,比例为9:1
split_index1 = np.random.choice(
    a=["train", "test"], replace=True, p=[0.9, 0.1], size=20
)
split_index1

array(['train', 'test', 'train', 'train', 'train', 'train', 'train',
       'train', 'train', 'train', 'train', 'test', 'train', 'train',
       'train', 'train', 'train', 'train', 'train', 'train'], dtype='<U5')

In [42]:
# 查看数据集spa.txt的行数
!wc -l datasets/spa-eng_extracted/spa-eng/spa.txt

  118964 datasets/spa-eng_extracted/spa-eng/spa.txt


In [ ]:
from pathlib import Path  # 导入Path类
from torch.utils.data import Dataset, DataLoader  # 导入Dataset类和DataLoader类


class LangPairDataset(Dataset):
    """
    语言对数据集类，用于加载西班牙语-英语平行语料。
    支持缓存机制，避免每次重新预处理原始文本。
    """

    # 原始语料文件路径（spa-eng 平行句对，以制表符分隔）
    fpath = Path(r"./datasets/spa-eng_extracted/spa-eng/spa.txt")
    # 预处理后的缓存文件路径，加速二次加载
    cache_path = Path(r"./.cache/lang_pair.npy")
    # 预先生成 118964 条样本的 train/test 划分标记，90% 训练，10% 测试
    # 注意：随机种子未固定，每次重启脚本划分可能不同
    split_index = np.random.choice(
        a=["train", "test"], replace=True, p=[0.9, 0.1], size=118964
    )

    def __init__(self, mode="train", cache=False):
        """
        初始化数据集。
        参数:
            mode (str): 取 "train" 或 "test"，决定返回哪一部分数据。
            cache (bool): 为 True 时强制重新预处理并覆盖缓存；为 False 时优先读取已有缓存。
        """
        # 如果没有缓存或缓存文件不存在，则重新预处理原始文件
        if cache or not self.cache_path.exists():
            # 确保缓存目录存在
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            # 读取原始平行语料
            with open(self.fpath, "r", encoding="utf-8") as file:
                lines = file.readlines()  # 读取所有行
                # 每行按制表符分割成西班牙语和英语句对，并对每个句子做预处理
                lang_pair = [
                    [preprocess_sentence(w) for w in l.split("\t")] for l in lines
                ]
                # 解压成两个列表：trg 为目标语言（英语），src 为源语言（西班牙语）
                """分离出目标语言和源语言,如果让英语作为src,把trg和src位置交换即可"""
                trg, src = zip(*lang_pair)
                # 转成 numpy 数组，便于后续索引
                trg = np.array(trg)
                src = np.array(src)
                # 将处理好的句对存入缓存文件，格式为字典 {"trg": 目标语言数组, "src": 源语言数组}
                np.save(self.cache_path, {"trg": trg, "src": src})
        else:
            # 已有缓存，直接加载
            # 读取npy文件,allow_pickle=True允许读取字典
            lang_pair = np.load(self.cache_path, allow_pickle=True).item()
            trg = lang_pair["trg"]
            src = lang_pair["src"]

        # 根据 mode 筛选对应划分的句对
        self.trg = trg[self.split_index == mode]  # 按照index拿到训练集,标签语言--英语
        self.src = src[self.split_index == mode]  # 按照index拿到训练集,源语言--西班牙语

    def __getitem__(self, index):
        """
        按索引返回一个样本，格式为 (源语言句子, 目标语言句子)
        """
        return self.src[index], self.trg[index]

    def __len__(self):
        """
        返回数据集大小（句子对数量）
        """
        return len(self.src)


# 实例化训练集与测试集
train_ds = LangPairDataset(mode="train")
test_ds = LangPairDataset(mode="test")

In [ ]:
print("训练集长度:", len(train_ds))
print("验证集长度:", len(test_ds))
# 查看最后一个样本
print("源序列:{}\n目标:{}.".format(*train_ds[-1]))

训练集长度: 107263
验证集长度: 11701
源序列:<start> si quieres sonar como un hablante nativo , debes estar dispuesto a practicar diciendo la misma frase una y otra vez de la misma manera en que un musico de banjo practica el mismo fraseo una y otra vez hasta que lo puedan tocar correctamente y en el tiempo esperado . <end>
目标:<start> if you want to sound like a native speaker , you must be willing to practice saying the same sentence over and over in the same way that banjo players practice the same phrase over and over until they can play it correctly and at the desired tempo . <end>.


# Tokenizer

这里有两种处理方式,分别对应着encoder和decoder的word embedding 是否共享,这里实现不共享的方案

In [ ]:
from collections import Counter


# 从数据集ds中提取单词到索引的映射(word2idx)和索引到单词的映射(idx2word)
def get_word_idx(ds, mode="src", threshold=2):
    # 载入词表,看下单词长度,词表就像英语字典
    word2idx = {
        "[PAD]": 0,  # 填充符：用于将序列填充到相同长度
        "[BOS]": 1,  # 句子起始符：用于标识一个句子的开始
        "[UNK]": 2,  # 未知词符：用于表示未登录词或未出现在训练集中的词
        "[EOS]": 3,  # 句子结束符：用于标识一个句子的结束
    }
    idx2word = {value: key for key, value in word2idx.items()}  # 索引到单词的映射
    index = len(idx2word)  # 词表大小
    threshold = 1  # 词频阈值,小于该值的词将被认为是未知词,舍弃该token
    # 太大不要放入内存,避免内存溢出,word_list是训练集所有的词,变为一个列表,列表每个元素都是一个词
    word_list = " ".join(
        pair[0 if mode == "src" else 1]
        for pair in ds  # 遍历数据集,将所有句子合并为一个字符串
        # 0 if mode == "src" else 1 表示:如果mode是src,则取pair[0],否则取pair[1]
    ).split()  # 所有词的列表
    # print(word_list)
    counter = Counter(word_list)  # 词频统计, counter类似词典,key是词,value是词频
    print(f"{mode} word counter:", len(counter))

    # 统计出现次数大于等于threshold的词,并将其加入word2idx
    for word, count in counter.items():
        if count >= threshold:  # 词频大于等于阈值,加入词表
            word2idx[word] = index  # 加入词表,索引从4开始
            idx2word[index] = word  # 加入索引到单词的映射
            index += 1  # 索引递增,准备加入下一个词
    return word2idx, idx2word


src_word2idx, src_idx2word = get_word_idx(train_ds, mode="src")  # 源语言词表 西班牙语
trg_word2idx, trg_idx2word = get_word_idx(train_ds, mode="tgt")  # 目标语言词表 英语

src word counter: 23776
tgt word counter: 12529


In [ ]:
print("确保src_word2idx和src_idx2word的长度相等")
print("src_word2idx:", len(src_word2idx))
print("src_idx2word:", len(src_idx2word))
print("trg_word2idx:", len(trg_word2idx))
print("tgt_idx2word:", len(trg_idx2word))

确保src_word2idx和src_idx2word的长度相等
src_word2idx: 23780
src_idx2word: 23780
trg_word2idx: 12533
tgt_idx2word: 12533


In [ ]:
class Tokenizer:
    def __init__(
        self,
        word2idx,
        idx2word,
        max_len=500,
        bos_idx=1,
        eos_idx=3,
        unk_idx=2,
        pad_idx=0,
    ):
        self.word2idx = word2idx
        self.idx2word = idx2word
        self.max_len = max_len
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.unk_idx = unk_idx
        self.pad_idx = pad_idx

    # 编码文本列表,将文本转换为索引序列
    def encode(
        self,
        text_list,
        padding_first=False,
        add_bos=True,
        add_eos=True,
        return_mask=False,
    ):
        """
        text_liset是一个二维列表,里面的每一个元素是一个样本,样本里面是词的列表
        如果padding_first为True,则在样本前面填充0,否则在样本后面填充0   [填补到前面效果更好]
        如果add_bos为True,则在样本前面添加bos_id,否则不添加
        如果add_eos为True,则在样本后面添加eos_id,否则不添加
        如果return_mask为True,则返回一个与text_list形状相同的mask矩阵,其中1表示有效位置,0表示填充位置
        (return_mask返回mask(掩码),用于attention计算,有效位置为1,填充位置为0)
        """
        # 当前批次的最大长度,取max_len和样本最大长度的较小值
        max_length = min(
            self.max_len, add_eos + add_bos + max([len(text) for text in text_list])
        )
        indices_list = []
        for text in text_list:
            # 如果此表中没有此词,则将其替换为unk_id,indices是词的索引列表
            indices = [
                self.word2idx.get(word, self.unk_idx)
                for word in text[: max_length - add_eos - add_bos]
            ]

            if add_bos:
                indices = [self.bos_idx] + indices
            if add_eos:
                indices += [self.eos_idx]
            # padding加载前面,超参可以调
            if padding_first:
                indices += [self.pad_idx] * (max_length - len(indices))
            else:  # padding加载后面
                indices += [self.pad_idx] * (max_length - len(indices))
            indices_list.append(indices)
        input_ids = torch.tensor(indices_list)  # 将列表转换为张量(tensor)
        """
        mask:掩码矩阵,用于去除padding的影响, [屏蔽padding的填充]
            maskes是一个与input_ids形状相同的张量,其中1表示有效位置,0表示填充位置
                即 0代表token,1代表padding. 
            mask用于去除padding的影响
        """
        masks = (input_ids == self.pad_idx).to(dtype=torch.int64)
        return input_ids if not return_mask else (input_ids, masks)

    # 解码函数,将索引列表转换为文本列表
    def decode(
        self,
        indices_list,
        remove_bos=True,
        remove_eos=True,
        remove_pad=True,
        split=False,  # 是否将文本列表转换为字符串列表
    ):
        """
        indices_list是一个二维列表,里面的每一个元素是一个样本,样本里面是词的索引列表
        如果remove_bos为True,则在样本前面移除bos_id,否则不移除
        如果remove_eos为True,则在样本后面移除eos_id,否则不移除
        如果remove_pad为True,则在样本中移除pad_id,否则不移除
        如果return_str为True,则返回一个二维字符串列表,否则返回一个二维索引列表
        """
        text_list = []
        for indices in indices_list:
            text = []
            for index in indices:
                word = self.idx2word.get(
                    index, "[UNK]"
                )  # 如果词表中没有这个词,就用unk_idx代替
                if remove_bos and word == "[BOS]":  # 如果移除bos,则跳过bos
                    continue
                if remove_eos and word == "[EOS]":  # 如果达到eos,则在遇到eos时停止
                    break
                if remove_pad and word == "[PAD]":  # 如果达到pad,就结束
                    continue
                text.append(word)  # 将词添加到文本列表中
            # 把列表中的单词拼接,变成一个句子,split=True时,返回列表,split=False时,返回句子
            #返回列表的目的是为了画注意力热力图
            text_list.append(
                " ".join(text) if not split else text
            )  # 将文本列表添加到文本列表中
        return text_list


# 两个相对应的tokenizer的好处是embedding的参数量减少
src_tokenizer = Tokenizer(
    word2idx=src_word2idx, idx2word=src_idx2word
)  # 源语言tokenizer
trg_tokenizer = Tokenizer(
    word2idx=trg_word2idx, idx2word=trg_idx2word
)  # 目标语言tokenizer

raw_text = [
    "hello world".split(),
    "tokenize text datas with batch".split(),
    "this is a test".split(),
]
indices, mask = trg_tokenizer.encode(
    raw_text, padding_first=True, add_bos=True, add_eos=True, return_mask=True
)
print(
    "encode 方法的作用是将原始文本（已分词为单词列表）转换为模型能够处理的数值化张量，同时根据需要添加特殊标记、处理填充和生成掩码。"
)
print("raw text" + "-" * 20)
for raw in raw_text:
    print(raw)
print(
    "mask" + "-" * 20 + "\nmask中显示为1的部分是padding（填充）部分,0表示有效位置"
)  # padding_first为True,则在样本前面填充0
for m in mask:
    print(m)
print(
    "indices"
    + "-" * 20
    + "\nindices中显示为0的部分是padding（填充）部分,\n1表示<bos>,3表示<eos>"
)
for index in indices:
    print(index)

encode 方法的作用是将原始文本（已分词为单词列表）转换为模型能够处理的数值化张量，同时根据需要添加特殊标记、处理填充和生成掩码。
raw text--------------------
['hello', 'world']
['tokenize', 'text', 'datas', 'with', 'batch']
['this', 'is', 'a', 'test']
mask--------------------
mask中显示为1的部分是padding（填充）部分,0表示有效位置
tensor([0, 0, 0, 0, 1, 1, 1])
tensor([0, 0, 0, 0, 0, 0, 0])
tensor([0, 0, 0, 0, 0, 0, 1])
indices--------------------
indices中显示为0的部分是padding（填充）部分,
1表示<bos>,3表示<eos>
tensor([   1,  302, 3228,    3,    0,    0,    0])
tensor([   1,    2, 3894,    2,  546,    2,    3])
tensor([   1,  122,  237,  109, 1272,    3,    0])


In [54]:
# 将索引序列解码为文本，保留BOS、EOS和PAD标记
decode_text = trg_tokenizer.decode(
    indices.tolist(), remove_bos=False, remove_eos=False, remove_pad=False
)
# 打印分隔线，提示开始输出解码文本
print(
    "decode text"
    + "-" * 20
    + "\ndecode方法用于将索引列表（即编码后的数字序列）转换回原始的文本（或单词列表）"
)
# 遍历并逐行打印解码后的文本
for decode in decode_text:
    print(decode)
    
print("-" * 20)
print("实际上看到的输出应该是:")
# 将索引序列解码为文本，保留BOS、EOS和PAD标记
decode_text = trg_tokenizer.decode(
    indices.tolist(), remove_bos=True, remove_eos=True, remove_pad=True
)
# 打印分隔线，提示开始输出解码文本
print(
    "decode text"
    + "-" * 20
    + "\ndecode方法用于将索引列表（即编码后的数字序列）转换回原始的文本（或单词列表）"
)
# 遍历并逐行打印解码后的文本
for decode in decode_text:
    print(decode)

decode text--------------------
decode方法用于将索引列表（即编码后的数字序列）转换回原始的文本（或单词列表）
[BOS] hello world [EOS] [PAD] [PAD] [PAD]
[BOS] [UNK] text [UNK] with [UNK] [EOS]
[BOS] this is a test [EOS] [PAD]
--------------------
实际上看到的输出应该是:
decode text--------------------
decode方法用于将索引列表（即编码后的数字序列）转换回原始的文本（或单词列表）
hello world
[UNK] text [UNK] with [UNK]
this is a test


# DataLoader

In [ ]:
def collate_fct(batch):
    # 从 batch 中提取源语言句子列表，每个元素是输入序列的原始文本
    src_words = [pair[0] for pair in batch]  # 获取batch内第0列进行分词,赋给src_words
    # 从 batch 中提取目标语言句子列表，用于后续解码器输入与标签生成
    tgt_words = [pair[1] for pair in batch]  # 获取batch内第1列进行分词,赋给tgt_words

    # 对源语言句子进行编码，得到模型所需的张量及对应的 attention mask
    # padding_first=True 表示在序列左侧填充，保证右对齐
    # add_bos=True / add_eos=True 在序列首尾添加起始符与结束符
    # return_mask=True 返回用于屏蔽 padding 的布尔 mask
    encoder_inputs, encoder_input_mask = src_tokenizer.encode(
        src_words, padding_first=True, add_bos=True, add_eos=True, return_mask=True
    )

    # 对目标语言句子编码，生成解码器的输入序列
    # padding_first=False 表示在右侧进行填充，符合自回归生成习惯
    # add_bos=True 在序列开头添加起始符，使解码器能够开始生成
    # add_eos=False 不在输入末尾加结束符，避免模型在训练时提前看到终止信号
    # return_mask=False 此处不需要返回 mask，因为解码器输入的 mask 会在后续由模型内部处理
    decoder_inputs = trg_tokenizer.encode(
        tgt_words, padding_first=False, add_bos=True, add_eos=False, return_mask=False
    )

    # 再次对目标语言句子编码，生成训练标签
    # 与 decoder_inputs 相比，这里 add_bos=False / add_eos=True
    # 保证标签序列与输入序列错开一个位置，实现“下一 token 预测”目标
    # return_mask=True 返回 mask，用于在计算损失时忽略 padding 部分
    decoder_labels, decoder_labels_mask = trg_tokenizer.encode(
        tgt_words, padding_first=False, add_bos=False, add_eos=True, return_mask=True
    )

    # 将所有张量转移到指定计算设备（如 GPU）并封装成字典返回
    return {
        "encoder_inputs": encoder_inputs.to(device=device),
        "encoder_input_mask": encoder_input_mask.to(device=device),
        "decoder_inputs": decoder_inputs.to(device=device),
        "decoder_labels": decoder_labels.to(device=device),
        "decoder_labels_mask": decoder_labels_mask.to(
            device=device
        ),  # mask用于去除padding的影响,计算loss时用
    }  # 当前返回的数据较多时,用dict返回比较合理

In [60]:
sample_dl=DataLoader(train_ds,batch_size=2,shuffle=True,collate_fn=collate_fct)

#两次执行这个代码不一样,因为每次执行都会shuffle
for batch in sample_dl:
    for key, value in batch.items():
        print(key,'\n' ,value,'\n'+'-'*50)
        
    break


encoder_inputs 
 tensor([[    1,     2,  8537,     2,    97, 22594,     2,     2,     2,    15,
             2,     2, 17113,  6894,     2,  2261, 18163,  6894, 15816,  8537,
            97,     2,     2,  1613,  9782,     2,  8537,  1613,  6644, 22594,
          6894,     2,  6894,  8537,     2,  1613,     2,    17,     2,     2,
          6894, 15816, 19162,     2,     3],
        [    1,     2,  8537,     2,    97, 22594,     2,     2,     2, 14027,
          6894, 23501,    97, 15816,     2,    97,     2,     2, 17113,  8537,
             2,  9782,    97, 15816,  1613,  8537,     2,     6,     2,     2,
          6894, 15816, 19162,     2,     3,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0]]) 
--------------------------------------------------
encoder_input_mask 
 tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 

# 设置模型

In [67]:
# 编码器类：将输入序列编码为固定长度的向量表示
#在RNN的基础上增加了注意力机制
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        hidden_dim=1024,
        num_layers=1,
    ):
        '''
        编码器类：将输入序列编码为固定长度的向量表示
            vocab_size: 词汇表大小，即输入序列中可能出现的不同词的数量
            embedding_dim: 词嵌入维度，即每个词被映射为的向量维度
            hidden_dim: RNN 隐藏层维度，即 RNN 单元内部状态的维度
            num_layers: RNN 层数，即 RNN 单元的堆叠数量
        '''
        super().__init__()
        # 词嵌入层：将输入的词索引映射为指定维度的向量
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # GRU 循环神经网络：对嵌入序列进行编码，提取时序特征
        '''
        GRU 层：对嵌入序列进行循环编码。参数说明：
            embedding_dim:输入特征维度（即嵌入向量维度）。
            hidden_dim:隐藏状态维度。
            num_layer:堆叠的 GRU 层数。
            batch_first=True:要求输入张量的形状为 (batch_size, seq_len, input_size)，输出同样为 (batch_size, seq_len, hidden_dim)
        '''
        self.rnn = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
    # 前向传播函数：将输入序列编码为固定长度的向量表示
    def forward(self, encoder_inputs):
        '''
        前向传播函数：将输入序列编码为固定长度的向量表示
            encoder_inputs: 输入序列，形状为 (batch_size, seq_len)，包含了输入序列的词索引
        返回：
            sqe_output: 序列输出，形状为 (batch_size, seq_len, hidden_dim)，包含了每个时间步的隐藏状态
            hidden: 最终隐藏状态，形状为 (num_layers, batch_size, hidden_dim)，包含了序列的全局信息
        '''
        # 将输入的词索引序列转换为嵌入向量序列
        embeds = self.embedding(encoder_inputs)
        # 经过 GRU 得到序列输出和最终隐藏状态
        sqe_output, hidden = self.rnn(embeds)
        return sqe_output, hidden

In [68]:
#把上面的Encoder写成一个例子,看看输出shape
encoder=Encoder(vocab_size=100,embedding_dim=32,hidden_dim=1024,num_layers=4)
encoder_inputs=torch.randint(0,100,(2,50))
encoder_outputs,hidden=encoder(encoder_inputs)
print(encoder_outputs.shape)
print(hidden.shape)
print(encoder_outputs[:,-1:])
print(hidden[-1:,:])#取最后一层的隐藏状态

torch.Size([2, 50, 1024])
torch.Size([4, 2, 1024])
tensor([[[-0.0500,  0.0434, -0.0177,  ...,  0.0356,  0.0210, -0.0096]],

        [[-0.0390,  0.0337, -0.0155,  ...,  0.0419, -0.0024,  0.0044]]],
       grad_fn=<SliceBackward0>)
tensor([[[-0.0500,  0.0434, -0.0177,  ...,  0.0356,  0.0210, -0.0096],
         [-0.0390,  0.0337, -0.0155,  ...,  0.0419, -0.0024,  0.0044]]],
       grad_fn=<SliceBackward0>)


# BahdanauAttention 类：实现 Bahdanau 注意力机制

> Bahdanau 注意力机制（也称为加性注意力），是序列到序列模型中常用的注意力形式，允许解码器在生成每个输出时动态关注编码器输出的不同部分。

BahdanauAttention公式:

score=FC(tanh(FC(EO)+FC(H)))  FC(EO)的FC是Wk,FC(H)的FC是Wq, 最外面的FC是V

attention_weights=softmax(score,axis=1)

context=sum(attention_weights*EO,axis=1)# 对EO做加权和,得到上下文向量
    

In [73]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        self.Wk = nn.Linear(hidden_dim, hidden_dim)  # 对keys做运算,encoder的输出EO
        self.Wq = nn.Linear(hidden_dim, hidden_dim)  # 对query做运算,decoder的隐状态H
        self.V = nn.Linear(hidden_dim, 1)  # V矩阵

    # 前向传播
    def forward(self, query, keys, values, attn_mask=None):
        """
        query: Decoder 的隐状态 (batch_size, hidden_dim)
        keys: Encoder 的输出 (batch_size, seq_len, hidden_dim)
        values: Encoder 的输出 (batch_size, seq_len, hidden_dim)
        attn_mask: 注意力掩码 (batch_size, seq_len)
        """
        # hidden_with_time_axis shape: (batch_size, 1, hidden_dim)
        # 为了计算分数，我们需要增加一个维度,
        # 将query的维度从(batch_size, hidden_dim)转换为(batch_size, 1, hidden_dim)
        hidden_with_time_axis = query.unsqueeze(-2)

        # query.shape: (batch_size, 1, hidden_dim) --> 通过unsqueeze(-2)转换为(batch_size, 1, hidden_dim)
        # keys.shape=(batch_size, seq_len, hidden_dim) --> 通过unsqueeze(-2)转换为(batch_size, seq_len, 1, hidden_dim)
        # values.shape=(batch_size, seq_len, hidden_dim) --> 通过unsqueeze(-2)转换为(batch_size, seq_len, 1, hidden_dim)

        # 计算分数
        # score shape: (batch_size, seq_len, 1)
        # 我们得到的分数是在最后一个轴上应用tanh(W1(keys) + W2(hidden_with_time_axis)) * V
        score = self.V(F.tanh(self.Wk(keys) + self.Wq(hidden_with_time_axis)))

        # 如果提供了注意力掩码，我们需要将其应用到分数中
        if attn_mask is not None:
            # 这个mask是encoder_input_mask,用于mask掉padding的部分,让padding部分scores为0
            # attn_mask shape: (batch_size, seq_len) --> 通过unsqueeze(-1)转换为(batch_size, seq_len, 1)
            attn_mask = (
                attn_mask.unsqueeze(-1)
            ) * -1e16  # 在最后增加一个维度,将mask转换为(batch_size, seq_len, 1)
            # 将掩码应用到分数中，将填充位置的分数设置为一个非常小的值（如负无穷）
            score += attn_mask
        score = F.softmax(score, dim=-2)

        # 对每个词的score和对应的value做乘法,然后在seq_len维度上求和,得到context_vector
        context_vector = torch.mul(score, values).sum(dim=-2)

        return context_vector, score


if __name__ == "__main__":
    # 测试 BahdanauAttention
    attention_layer = BahdanauAttention(hidden_dim=1024)
    query = torch.randn(2, 1024)# Decoder的隐藏状态
    keys = torch.randn(2, 5, 1024)#Encoder的输出EO
    values = torch.randn(2, 5, 1024)#Encoder的输出EO
    attn_mask = torch.randint(0, 2, (2, 5))
    context_vector, scores = attention_layer(query, keys, values, attn_mask)
    print(context_vector.shape)
    print(scores.shape)
    print(scores)

torch.Size([2, 1024])
torch.Size([2, 5, 1])
tensor([[[0.2861],
         [0.4654],
         [0.0000],
         [0.0000],
         [0.2485]],

        [[0.0000],
         [0.0000],
         [0.0000],
         [0.4580],
         [0.5420]]], grad_fn=<SoftmaxBackward0>)


In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        hidden_dim=1024,
        attention=None,
        num_layers=1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(
            embedding_dim + hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.6)  # 0.6可以调整的超参数
        self.attention = BahdanauAttention(hidden_dim)

    def forward(self, decoder_input, hidden, encoder_outputs, mask=None):
        assert (
            len(decoder_input) == 2 and decoder_input.shape[-1] == 1
        ), f"decoder_input.shape={decoder_input.shape} is not vaild"
        assert len(hidden) == 2, f"hidden.shape={hidden.shape} is not vaild"
        assert (
            len(encoder_outputs) == 3
        ), f"encoder_outputs.shape={encoder_outputs.shape} is not vaild"

        context_vector, attention_score = self.attention(
            query=hidden,
            key=encoder_outputs,
            value=encoder_outputs,
            attention_mask=mask,
        )

        embes = self.embedding(decoder_input)
        embes = torch.cat([context_vector.unsqueeze(-2), embes], dim=-1)

        seq_output, hidden = self.gru(embes)
        logits = self.fc(self.dropout(seq_output))
        return logits, hidden, attention_score